In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 4, Finished, Available, Finished, False)

In [3]:
# Load the cleaned Silver data from the Lakehouse
silver_df = spark.read.table("silver_flipkart_sales")

# Create a temporary view so we can easily query it using standard SQL
silver_df.createOrReplaceTempView("silver")



StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 5, Finished, Available, Finished, False)

In [4]:
#creating gold_sales_fact
gold_sales_fact = spark.sql("""
    SELECT 
        OrderID,
        OrderDate,
        DeliveryDate,
        CustomerID,
        Location,
        DeliveryType,
        ProductCategory,
        SubCategory,
        Product,
        UnitPrice,
        ShippingFee,
        OrderQuantity,
        SalePrice,
        Status,
        Reason,
        Rating,
        DeliveryDays,
        TotalRevenue,
        DiscountAmount,
        DiscountPct,
        IsReturned,
        OrderYear,
        OrderMonth
    FROM silver
""")

gold_sales_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_sales_fact")

print("Updated Gold Sales Fact table saved successfully!")

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 6, Finished, Available, Finished, False)

Updated Gold Sales Fact table saved successfully!


In [5]:
# Save the fact table to the Gold layer
gold_sales_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_sales_fact")

print("Gold Sales Fact table saved successfully!")
print("Rows saved:", gold_sales_fact.count())

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 7, Finished, Available, Finished, False)

Gold Sales Fact table saved successfully!
Rows saved: 19028


In [6]:
# Create the Gold Date Dimension Table
gold_dim_date = spark.sql("""
    SELECT DISTINCT
        OrderDate AS DateKey,
        YEAR(OrderDate) AS Year,
        MONTH(OrderDate) AS Month,
        DATE_FORMAT(OrderDate, 'MMMM') AS MonthName,
        QUARTER(OrderDate) AS Quarter,
        DAYOFWEEK(OrderDate) AS DayOfWeek,
        DATE_FORMAT(OrderDate, 'EEEE') AS DayName
    FROM silver
""")

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 8, Finished, Available, Finished, False)

In [7]:
gold_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_date")

print("Gold Date Dimension table saved successfully!")
print("Rows saved:", gold_dim_date.count())

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 9, Finished, Available, Finished, False)

Gold Date Dimension table saved successfully!
Rows saved: 2191


In [8]:
# Create the Gold Product Dimension Table
gold_dim_product = spark.sql("""
    SELECT DISTINCT
        Product, 
        SubCategory, 
        ProductCategory, 
        UnitPrice
    FROM silver
""")

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 10, Finished, Available, Finished, False)

In [9]:
gold_dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_product")

print("Gold Product Dimension table saved successfully!")
print("Rows saved:", gold_dim_product.count())

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 11, Finished, Available, Finished, False)

Gold Product Dimension table saved successfully!
Rows saved: 6255


In [10]:
# Create the Gold Monthly KPI Table
gold_monthly_kpi = spark.sql("""
    SELECT
        OrderYear,
        OrderMonth,
        ProductCategory,
        COUNT(DISTINCT OrderID) AS TotalOrders,
        SUM(TotalRevenue) AS TotalRevenue,
        SUM(OrderQuantity) AS TotalUnitsSold,
        AVG(Rating) AS AvgRating,
        SUM(IsReturned) AS TotalReturns,
        ROUND(SUM(IsReturned) * 100.0 / COUNT(*), 2) AS ReturnRatePct
    FROM silver
    GROUP BY OrderYear, OrderMonth, ProductCategory
""")

gold_monthly_kpi.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_monthly_kpi")

print("Gold Monthly KPI table saved successfully!")
print("Rows saved:", gold_monthly_kpi.count())

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 12, Finished, Available, Finished, False)

Gold Monthly KPI table saved successfully!
Rows saved: 360


In [11]:
# Create gold_dim_customer
gold_dim_customer = spark.sql("""
    SELECT DISTINCT
        CustomerID,
        CustomerAge,
        CustomerGender
    FROM silver
    WHERE CustomerID IS NOT NULL
""")

gold_dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_customer")

print("Gold Customer Dimension saved successfully!")
print("Total unique customers:", gold_dim_customer.count())

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 13, Finished, Available, Finished, False)

Gold Customer Dimension saved successfully!
Total unique customers: 19028


In [12]:
# Create gold_dim_location
gold_dim_location = spark.sql("""
    SELECT DISTINCT
        Location,
        Zone
    FROM silver
    WHERE Location IS NOT NULL
""")

gold_dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_location")

print("Gold Location Dimension saved successfully!")
print("Total unique locations:", gold_dim_location.count())

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 14, Finished, Available, Finished, False)

Gold Location Dimension saved successfully!
Total unique locations: 86


In [13]:
display(gold_sales_fact)

StatementMeta(, d2e76354-3201-4abd-b931-6531f12be812, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 59386a64-0cd4-4b98-b448-4025a883e2a8)

In [1]:
# 1. Read the current Gold Fact table
gold_df = spark.read.table("gold_sales_fact")

# 2. Force-drop the specific columns
clean_gold_df = gold_df.drop("CustomerAge", "CustomerGender", "Zone")

# 3. Overwrite the Lakehouse table with the clean version
clean_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_sales_fact")

print("Lakehouse table successfully cleaned!")

StatementMeta(, 3e7b9964-a7af-44d9-8cd9-f2b906e9d0dd, 3, Finished, Available, Finished, False)

Lakehouse table successfully cleaned!
